In [5]:
!pip uninstall -y tensorflow keras keras-nlp
!pip install tensorflow==2.19.0 keras-nlp==0.14.0 --quiet

Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: keras-nlp 0.21.1
Uninstalling keras-nlp-0.21.1:
  Successfully uninstalled keras-nlp-0.21.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 73.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [6]:
import os
import typing
from tqdm import tqdm
import glob
import pandas as pd
import tensorflow as tf
import numpy as np
import json
import keras
from keras.utils import  Sequence
import keras_nlp
os.path.exists('/kaggle/input/datasets/akami14/drive-redused/YaCupTrain') #/kaggle/input/drive-redused/YaCupTrain

True

In [7]:
print(tf.__version__)

2.19.0


In [9]:
print(keras.__version__)

3.10.0


In [10]:
ROOT_DATA_FOLDER = r"/kaggle/input/datasets/akami14/drive-redused"

TRAIN_DATASET_PATH = r'/kaggle/input/datasets/akami14/drive-redused/YaCupTrain'

#os.path.join(ROOT_DATA_FOLDER, r"YaCupTrain")

TEST_DATASET_PATH = os.path.join(ROOT_DATA_FOLDER, r"YaCupTest")
label_columns = ['x', 'y', 'yaw']

# Load all ids of a dataset

def read_testcase_ids(dataset_path: str):
    ids = [int(case_id) for case_id in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, case_id))]
    return ids

ids = read_testcase_ids(TRAIN_DATASET_PATH)
len(ids)

train_ids = np.random.choice(ids, size=round(0.70*len(ids)), replace=False, p=None)
test_ids = [el for el in ids if el not in train_ids]


class DataFilePaths:
    def __init__(self, testcase_path: str):
        self.testcase_path = testcase_path

    def localization(self):
        return os.path.join(self.testcase_path, 'localization.csv')

    def control(self):
        return os.path.join(self.testcase_path, 'control.csv')

    def metadata(self):
        return os.path.join(self.testcase_path, 'metadata.json')

    # exists only for test_dataset
    def requested_stamps(self):
        return os.path.join(self.testcase_path, 'requested_stamps.csv')


def read_localization(localization_path: str):
    return pd.read_csv(localization_path)

def read_control(control_path):
    return pd.read_csv(control_path)

def read_metadata(metadata_path: str):
    with open(metadata_path, 'r') as f:
        data = json.load(f)
    return data

def read_requested_stamps(requested_stamps_path: str):
    return pd.read_csv(requested_stamps_path)

def read_testcase(dataset_path: str, testcase_id: str, is_test: bool = False):
    testcase_path = os.path.join(dataset_path, str(testcase_id))
    data_file_paths = DataFilePaths(testcase_path)

    testcase_data = {}
    testcase_data['localization'] = read_localization(data_file_paths.localization())
    testcase_data['control'] = read_control(data_file_paths.control())
    testcase_data['metadata'] = read_metadata(data_file_paths.metadata())
    if is_test:
        testcase_data['requested_stamps'] = read_requested_stamps(data_file_paths.requested_stamps())

    return testcase_data


def read_testcases(dataset_path: str, is_test: bool = False, testcase_ids: typing.Iterable[int] = None):
    result = {}
    if testcase_ids is None:
        testcase_ids = read_testcase_ids(dataset_path)

    for testcase_id in tqdm(testcase_ids):
        testcase = read_testcase(dataset_path, testcase_id, is_test=is_test)
        result[testcase_id] = testcase
    return result
    
train_dataset = read_testcases(TRAIN_DATASET_PATH)
len(train_dataset)

100%|██████████| 3884/3884 [01:05<00:00, 59.37it/s]


3884

# Split

In [11]:
def train_val_split(cars_data, val_ratio=0.2, seed=42):

    car_ids = list(cars_data.keys())
    rng = np.random.default_rng(seed)
    rng.shuffle(car_ids)

    split_idx = int(len(car_ids) * (1 - val_ratio))

    train_ids = car_ids[:split_idx]
    val_ids = car_ids[split_idx:]

    train_data = {cid: cars_data[cid] for cid in train_ids}
    val_data = {cid: cars_data[cid] for cid in val_ids}
    return train_data, val_data

train_cars_data, val_cars_data = train_val_split(train_dataset)



In [13]:
class RawTrajectorySequence(Sequence):
    def __init__(
        self,cars_data, history,
        horizon,batch_size,stride=1,shuffle=True):

        self.history = history
        self.horizon = horizon
        self.batch_size = batch_size
        self.stride = stride
        self.shuffle = shuffle
        self._prepare(cars_data)
        self.on_epoch_end()


    def _prepare(self, cars_data):

        self.data = {}
        self.index = []
        #Подготовка данных (CPU, один раз)

        for car_id, car in cars_data.items():
            df = pd.merge_asof(
                car["localization"].sort_values("stamp_ns"),
                car["control"].sort_values("stamp_ns"),
                on="stamp_ns",
                direction="nearest").dropna().reset_index(drop=True)

            self.data[car_id] = {
                "xy": df[["x", "y"]].to_numpy(np.float32),
                "z": df["z"].to_numpy(np.float32),
                "roll": df["roll"].to_numpy(np.float32),
                "pitch": df["pitch"].to_numpy(np.float32),
                "yaw": df["yaw"].to_numpy(np.float32),
                "acc": df["acceleration_level"].to_numpy(np.float32),
                "steer": df["steering"].to_numpy(np.float32),}

            T = len(df)
            for t0 in range(self.history, T - self.horizon, self.stride):
                self.index.append((car_id, t0))

    def __len__(self):
        return len(self.index) // self.batch_size


    def __getitem__(self, idx):

        batch = self.index[
            idx * self.batch_size : (idx + 1) * self.batch_size]
        B = self.batch_size
        X = np.zeros((B, self.history, 6), np.float32)
        y = np.zeros((B, self.horizon, 2), np.float32)
        xy0 = np.zeros((B, 2), np.float32)
        yaw0 = np.zeros((B,), np.float32)
        for i, (car_id, t0) in enumerate(batch):
            d = self.data[car_id]
            X[i] = np.stack([
                d["xy"][t0 - self.history : t0, 0],
                d["xy"][t0 - self.history : t0, 1],
                d["z"][t0 - self.history : t0],
                d["roll"][t0 - self.history : t0],
                d["pitch"][t0 - self.history : t0],
                d["acc"][t0 - self.history : t0],], axis=1)
            y[i] = d["xy"][t0 : t0 + self.horizon]
            xy0[i] = d["xy"][t0 - 1]
            yaw0[i] = d["yaw"][t0 - 1]
        return (X, xy0, yaw0), y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.index)


class TrajectoryPreprocess(tf.keras.layers.Layer):

    def call(self, inputs):
        X, xy0, yaw0 = inputs
        dx = X[..., 0] - xy0[:, None, 0]
        dy = X[..., 1] - xy0[:, None, 1]
        cos_y = tf.cos(yaw0)[:, None]
        sin_y = tf.sin(yaw0)[:, None]
        x_loc =  cos_y * dx + sin_y * dy
        y_loc = -sin_y * dx + cos_y * dy
        dx = x_loc - tf.pad(x_loc[:, :-1], [[0, 0], [1, 0]])
        dy = y_loc - tf.pad(y_loc[:, :-1], [[0, 0], [1, 0]])
        roll = X[..., 3]
        pitch = X[..., 4]
        features = tf.concat([
            dx[..., None],
            dy[..., None],
            X[..., 2:3],
            tf.sin(roll)[..., None],
            tf.cos(roll)[..., None],
            tf.sin(pitch)[..., None],
            tf.cos(pitch)[..., None],
            X[..., 5:6], ], axis=-1)
        return features


In [14]:
class FastTrajectoryWindowSequence(Sequence):

    def __init__(
        self,
        cars_data: dict,
        history: int,
        horizon: int,
        batch_size: int,
        stride: int = 1,
        stats: dict | None = None,
        shuffle: bool = True):

        self.history = history
        self.horizon = horizon
        self.batch_size = batch_size
        self.stride = stride
        self.shuffle = shuffle
        self.stats = stats or self._default_stats()
        self._prepare_arrays(cars_data)
        self.on_epoch_end()


    def _default_stats(self):
        return {
            "acc_min": -5.0,
            "acc_max":  5.0,
            "acc_mean": 0.0,
            "acc_std":  1.0,
            "steer_min": -1.5,
            "steer_max":  1.5,
            "steer_mean": 0.0,
            "steer_std":  0.5,}


    def _prepare_arrays(self, cars_data):
        self.data = {}
        self.index = []

        for car_id, car in cars_data.items():
            df = pd.merge_asof(
                car["localization"].sort_values("stamp_ns"),
                car["control"].sort_values("stamp_ns"),
                on="stamp_ns",
                direction="nearest").dropna().reset_index(drop=True)

            # --- numpy arrays ---
            x = df["x"].to_numpy(np.float32)
            y = df["y"].to_numpy(np.float32)
            z = df["z"].to_numpy(np.float32)
            yaw = df["yaw"].to_numpy(np.float32)

            roll = df["roll"].to_numpy(np.float32)
            pitch = df["pitch"].to_numpy(np.float32)

            roll_sin, roll_cos = np.sin(roll), np.cos(roll)
            pitch_sin, pitch_cos = np.sin(pitch), np.cos(pitch)

            acc = np.clip(
                df["acceleration_level"].to_numpy(np.float32),
                self.stats["acc_min"], self.stats["acc_max"])
            steer = np.clip(
                df["steering"].to_numpy(np.float32),
                self.stats["steer_min"], self.stats["steer_max"])

            acc = (acc - self.stats["acc_mean"]) / (self.stats["acc_std"] + 1e-6)
            steer = (steer - self.stats["steer_mean"]) / (self.stats["steer_std"] + 1e-6)

            features = np.stack([
                z, roll_sin, roll_cos,
                pitch_sin, pitch_cos,
                acc, steer], axis=1)

            self.data[car_id] = {
                "features": features,   # (T, 7)
                "xy": np.stack([x, y], axis=1),
                "yaw": yaw}
            T = len(x)
            
            for t0 in range(self.history, T - self.horizon, self.stride):
                self.index.append((car_id, t0))


    def __len__(self):
        return len(self.index) // self.batch_size


    def __getitem__(self, idx):
        batch = self.index[
            idx * self.batch_size : (idx + 1) * self.batch_size]
        X = np.empty((self.batch_size, self.history, 9), np.float32)
        y = np.empty((self.batch_size, self.horizon, 2), np.float32)



        for i, (car_id, t0) in enumerate(batch):

            d = self.data[car_id]
            xy = d["xy"]
            yaw0 = d["yaw"][t0 - 1]

            cos_y, sin_y = np.cos(yaw0), np.sin(yaw0)
            past_xy = xy[t0 - self.history : t0]

            ref = xy[t0 - 1]

            dxy = past_xy - ref
            x_loc =  cos_y * dxy[:, 0] + sin_y * dxy[:, 1]
            y_loc = -sin_y * dxy[:, 0] + cos_y * dxy[:, 1]
            dx = np.diff(np.concatenate([[0.0], x_loc]))
            dy = np.diff(np.concatenate([[0.0], y_loc]))

            X[i] = np.concatenate([
                np.stack([dx, dy], axis=1),
                d["features"][t0 - self.history : t0]
            ], axis=1)
            fut_xy = xy[t0 : t0 + self.horizon] - ref
            x_f =  cos_y * fut_xy[:, 0] + sin_y * fut_xy[:, 1]
            y_f = -sin_y * fut_xy[:, 0] + cos_y * fut_xy[:, 1]
            y[i] = np.stack([x_f, y_f], axis=1)

        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.index)

In [15]:
train_seq = RawTrajectorySequence(cars_data=train_cars_data, history=20,horizon=10,batch_size=32)
test_seq = RawTrajectorySequence(cars_data=val_cars_data, history=20,horizon=10,batch_size=32)

In [19]:
def build_model(history, horizon):
    X = tf.keras.Input((history, 6))
    xy0 = tf.keras.Input((2,))
    yaw0 = tf.keras.Input(())
    x = TrajectoryPreprocess()((X, xy0, yaw0))
    x = tf.keras.layers.Dense(128)(x)
    #x=tf.nlp.models.TransformerEncoder(num_heads=5,key_dim=256)(x)
    x = keras_nlp.layers.TransformerEncoder(intermediate_dim=512, num_heads=8)(x)
    #x = tf.keras.layers.MultiHeadAttention(num_heads=5,key_dim=256, value_dim=256,dropout=0.1)(x)
    #x = tf.keras.layers.TransformerEncoder(num_heads=4,intermediate_dim=256)(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.RepeatVector(horizon)(x)
    out = tf.keras.layers.TimeDistributed(
    tf.keras.layers.Dense(2))(x)
    return tf.keras.Model([X, xy0, yaw0], out)



# Loss по Δx, Δy стабильнее на дальнем горизонте меньше drift
#Абсолютный MSE сильно растёт с горизонтом модель «боится» дальних шагов

def huber_xy(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    abs_error = tf.abs(error)

    quadratic = tf.minimum(abs_error, delta)
    linear = abs_error - quadratic

    loss = 0.5 * quadratic**2 + delta * linear
    return tf.reduce_sum(loss, axis=-1)

def delta_mse(y_true, y_pred):
    dy_true = y_true - tf.pad(y_true[:, :-1], [[0, 0], [1, 0], [0, 0]])
    dy_pred = y_pred - tf.pad(y_pred[:, :-1], [[0, 0], [1, 0], [0, 0]])
    return tf.reduce_mean(tf.square(dy_true - dy_pred))


def rms_loss(y_true, y_pred):
    return tf.sqrt(tf.reduce_sum(tf.square(y_true - y_pred), axis=-1) + 1e-8)



In [18]:
!pip freeze > requirements.txt

!pip freeze > requirements.txt

In [ ]:

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    model = build_model(history=20, horizon=10)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(3e-4),
        loss=huber_xy
        #delta_mse
        #tf.keras.losses.Huber(delta=1.0)
        #rms_loss
    )


model.fit(
    train_seq,
    validation_data=test_seq,
    epochs=50)

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0',)
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
Epoch 1/50
  5306/150626 ━━━━━━━━━━━━━━━━━━━━ 21:50 9ms/step - loss: 14064.0684